# Divvy Bike-Share Analytics — Chicago, Summer 2026
**2.5M trips · SQL (DuckDB) · Plotly dashboard**

Business questions:
1. Who rides — members or casual riders — and how do their trip patterns differ?
2. When does demand peak (hour, weekday)? What does that imply for rebalancing crews?
3. How is demand trending across the summer quarter?
4. Who uses e-bikes vs classic bikes?
5. Which stations are chronically imbalanced (bikes pile up / run out)?


In [1]:
import duckdb
con = duckdb.connect()
con.execute("""CREATE OR REPLACE VIEW trips AS
SELECT * FROM read_csv('data/2026*-divvy-tripdata/*.csv', header=true)
WHERE started_at >= '2026-06-01' AND started_at < '2026-09-01'""")
print(f"{con.execute('SELECT COUNT(*) FROM trips').fetchone()[0]:,} trips loaded")

2,499,714 trips loaded

## Q1 · Who rides?
Members take 59.5% of trips, but casual riders' trips run ~64% longer on average — leisure vs commute behavior.

In [1]:
q1 = con.execute("""
SELECT member_casual, COUNT(*) AS trips,
 ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (),1) AS pct,
 ROUND(AVG(date_diff('second',started_at,ended_at))/60,1) AS avg_min,
 ROUND(QUANTILE_CONT(date_diff('second',started_at,ended_at),0.5)/60,1) AS med_min
FROM trips WHERE ended_at>started_at GROUP BY 1 ORDER BY 2 DESC""").fetchdf()
q1

member_casual   trips  pct  avg_min  med_min
       member 1486496 59.5     12.8      9.2
       casual 1013218 40.5     20.9     11.6

<plotly figure>

## Q2 · When is demand highest?
Clear member commute spikes at 8 AM and 5 PM; casual demand builds through the afternoon and peaks at 5 PM too. Rebalancing crews should prioritize the 4–7 PM window.

In [1]:
q2 = con.execute("""
SELECT EXTRACT(HOUR FROM started_at)::INT AS hr, member_casual, COUNT(*) AS trips
FROM trips WHERE ended_at>started_at GROUP BY 1,2 ORDER BY 1,2""").fetchdf()
px.line(q2, x="hr", y="trips", color="member_casual",
        title="Trips by hour of day").update_traces(mode="lines+markers")

<plotly figure>

## Q3 · Commuters vs leisure
Members dominate Mon–Fri; on Saturday casual riders overtake members — the weekend leisure crowd.

In [1]:
q3 = con.execute("""
SELECT EXTRACT(ISODOW FROM started_at)::INT AS dow, member_casual, COUNT(*) AS trips
FROM trips WHERE ended_at>started_at GROUP BY 1,2 ORDER BY 1,2""").fetchdf()

<plotly figure>

## Q4 · Summer trend
Demand jumps ~14% from June to July, then holds steady through August — peak season is July/August.

In [1]:
q4 = con.execute("""
SELECT DATE_TRUNC('month',started_at)::DATE AS month, COUNT(*) AS trips
FROM trips WHERE ended_at>started_at GROUP BY 1 ORDER BY 1""").fetchdf()
q4

   month  trips
Jun 2026 762595
Jul 2026 869101
Aug 2026 868018

<plotly figure>

## Q5 · E-bike adoption
E-bikes dominate: 76% of casual trips and 70% of member trips. Charging/logistics capacity should assume an e-bike-majority fleet.

In [1]:
q5 = con.execute("""
SELECT rideable_type, member_casual, COUNT(*) AS trips
FROM trips WHERE ended_at>started_at GROUP BY 1,2 ORDER BY 2,3 DESC""").fetchdf()
q5

rideable_type member_casual   trips
electric_bike        casual  767038
 classic_bike        casual  246180
electric_bike        member 1044732
 classic_bike        member  441764

<plotly figure>

## Q6 · Rebalancing targets
Negative net outflow = bikes pile up (needs pickups); positive = bikes drain (needs drop-offs). **Franklin St & Monroe St** accumulates ~1,380 excess bikes; **Field Museum** loses ~1,020 — top candidates for scheduled rebalancing runs.

In [1]:
q6 = con.execute("""
WITH f AS (
 SELECT start_station_name AS s, COUNT(*) AS a, 0 AS b FROM trips
 WHERE start_station_name IS NOT NULL GROUP BY 1
 UNION ALL
 SELECT end_station_name AS s, 0 AS a, COUNT(*) AS b FROM trips
 WHERE end_station_name IS NOT NULL GROUP BY 1)
SELECT s AS station, SUM(a)-SUM(b) AS net_outflow
FROM f GROUP BY 1 HAVING SUM(a)+SUM(b)>=1000
ORDER BY ABS(SUM(a)-SUM(b)) DESC LIMIT 12""").fetchdf()
q6

                           station  net_outflow
           Franklin St & Monroe St      -1380.0
                      Field Museum       1022.0
             Wells St & Madison St       -888.0
         Clark St & Ida B Wells Dr        884.0
DuSable Lake Shore Dr & North Blvd       -791.0
             Michigan Ave & 8th St        788.0
         Columbus Dr & Randolph St        695.0
               Theater on the Lake       -666.0
 DuSable Lake Shore Dr & Monroe St        578.0
             Clark St & Newport St       -567.0
        Sheffield Ave & Addison St       -501.0
         Wacker Dr & Washington St       -501.0

<plotly figure>

## Q7 · Where do rides start?
Navy Pier leads by a wide margin (33.9K starts) — tourist/leisure demand, consistent with the casual-rider weekend pattern.

In [1]:
q7 = con.execute("""
SELECT start_station_name AS station, COUNT(*) AS trips
FROM trips WHERE start_station_name IS NOT NULL
GROUP BY 1 ORDER BY 2 DESC LIMIT 10""").fetchdf()
q7

                           station  trips
                         Navy Pier  33895
             Michigan Ave & Oak St  17547
DuSable Lake Shore Dr & North Blvd  17151
 DuSable Lake Shore Dr & Monroe St  16860
               Theater on the Lake  14285
            State St & Chicago Ave  11617
                   Millennium Park  11464
             Wells St & Concord Ln  11218
        Kingsbury St & Kinzie St 2  10874
                 Wells St & Elm St  10739

<plotly figure>

## Q8 · Rush-hour check
Member trips are shortest in the AM rush (8.5 min median) — classic commute behavior, confirming members are the commuter segment.

In [1]:
q8 = con.execute("""
SELECT CASE WHEN EXTRACT(HOUR FROM started_at) BETWEEN 7 AND 9 THEN 'AM rush (7-9)'
 WHEN EXTRACT(HOUR FROM started_at) BETWEEN 17 AND 19 THEN 'PM rush (17-19)'
 ELSE 'Off-peak' END AS period, member_casual,
 ROUND(QUANTILE_CONT(date_diff('second',started_at,ended_at),0.5)/60,1) AS med_min
FROM trips WHERE ended_at>started_at GROUP BY 1,2 ORDER BY 1,2""").fetchdf()
q8

         period member_casual  med_min
  AM rush (7-9)        casual      9.1
  AM rush (7-9)        member      8.5
       Off-peak        casual     11.9
       Off-peak        member      9.2
PM rush (17-19)        casual     11.8
PM rush (17-19)        member      9.8

<plotly figure>

## Key findings & recommendations
1. **Two distinct customer segments**: members (59.5% of trips) commute — AM/PM rush peaks, 8.5-min median AM trips; casual riders take 64% longer trips and dominate weekends. → *Tailor marketing: commuter passes for members, leisure/tourist bundles for casuals.*
2. **E-bike majority**: 70–76% of trips are e-bikes. → *Plan charging capacity and pricing around e-bikes, not classic bikes.*
3. **Rebalancing priorities**: Franklin St & Monroe St (+1,380 bike surplus) and Field Museum (−1,020 deficit) top the list. → *Schedule rebalancing runs around the 4–7 PM peak.*
4. **Seasonality**: July/August run ~14% above June. → *Staff and rebalance for peak summer.*
